# Class 9- RNAseq Analysis I

Start by Installing and Importing the necessary packages. Note that we will need to set some specific settings to get everything to run properly in Colaboratory.




Setting up some other Pandas and python presets

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import statsmodels.stats.multitest as smm
import matplotlib
import matplotlib.pyplot as plt
from statsmodels.formula.api import ols
import seaborn as sns
import csv
import os
import sys
import argparse
from pathlib import Path
from time import time

In [ ]:
import  scipy.signal.signaltools

def _centered(arr, newsize):
    # Return the center newsize portion of the array.
    newsize = np.asarray(newsize)
    currsize = np.array(arr.shape)
    startind = (currsize - newsize) // 2
    endind = startind + newsize
    myslice = [slice(startind[k], endind[k]) for k in range(len(endind))]
    return arr[tuple(myslice)]

scipy.signal.signaltools._centered = _centered

In [ ]:
from scipy.signal.signaltools import _centered as trim_centered

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

In [ ]:
pd.set_option('display.precision', 2)
pd.set_option('display.max_columns',10)

# We're also going to tell Jupyter to use inline plotting instead of notebook plotting
# It basically means you don't have to use plt.show() in every cell
%matplotlib inline

# and this command will allow multiple outputs from the same cell, rather than just the last one run
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

### Importing the data.

We will import a table of RNAseq counts in a sample x gene format.

Several steps have taken place to generate this file:

1. RNAseq raw reads data is comprised of fastq file(s), containing raw cDNA sequences and quality scores. Often paired.
2. RNAseq was aligned to a human reference genome.
    - Genome sequences (per chromosome), fasta format
    - Genome feature annotation file (gtf/gff) with gene locations
4. Alignment software produces bam/sam alignment files indicating where RNAseq reads map to the reference genome
5. Count table was then derived from alignment by counting number of unique reads mapping to each gene
   

In [ ]:
#importing the raw melanoma count table
data_dir = Path("~/LECTURE_MATERIALS/DataFiles")
CountsData = pd.read_csv(data_dir / 'melanoma_CountsRaw.csv',index_col = 0)

In [ ]:
counts = CountsData.loc[:,'A1BG':]
counts.head(12)

# Normalization and gene filtering

Before analysis, we will stop and take a look at some of the properties of the counts.

Each sample has a certain library size: the total number of reads that mapped to a gene for each sample. This is a technical effect we need to control for when looking at differential gene expression.

It is also a quality control metric: standard RNAseq recommendation is 20-30 million reads for a human sample for differential gene expression.

In [ ]:
sample_counts = counts.sum(axis=1)
ax = sample_counts.plot.bar()
ax.set_xlabel("Sample")
ax.set_ylabel("Total counts")

Genes also vary in read count: of course more highly expressed genes have higher counts, but longer genes also tend to have higher counts than shorter counts.

To gain a sense of the distribution of counts we can take a few different approaches, and we'll just use two here. In the first we extract the values from the pandas data frame and create a numpy array.  This allows us to plot a histrogram with 50 bins, where the counts are on the X axis, and the number of genes detected at that count level are displayed on the Y axis.

Here is a log-transformed histogram, we have to add a small positive count to the values to avoid taking the log of 0

In [ ]:
counts_values = counts.values.flatten()  # flatten the matrix into a 1D array
eps = 0.1
hist, bins = np.histogram(counts_values+eps, bins=50)
logbins = np.logspace(np.log10(bins[0]),np.log10(bins[-1]),len(bins))

plt.hist(counts_values, bins=logbins)
plt.title("Histogram of RNAseq counts")
plt.xlabel("Counts")
plt.ylabel("Frequency")
plt.xscale("log")


This histogram shows that many genes have counts close to 0, and there is a peak of more highly expressed genes around 100 - 10,000 counts

What if we plot per-sample? 

In [ ]:
#transpose the data and plot
temp =np.transpose(counts+eps)

matplotlib.rcParams['figure.figsize'] = [12, 10]
temp.plot.hist(subplots=True, bins=logbins, layout=(4,3))
plt.xscale("log")
plt.show()

## Filtering criteria

We can see that all samples have similar count distributions

We can also see that the greatest density of gene counts is around 1000, and that this is true for all samples.  

We want to remove all genes that are usually 0. But some genes might only be expressed in a single group of interest, so just filtering average counts may miss this.

Alternative approach: keep genes that have *reasonable* (e.g. >= 10) counts in at least the number of samples corresponding to our smallest group. 

In [ ]:
# Make a Data Frame of boolean values showing where there are read counts less than 1 (i.e. = 0)
passing_counts = counts >= 10

#How many samples each gene have enough counts?
n_passing_counts = passing_counts.sum()

#What is the size of the smallest group of interest?
group_counts = CountsData.loc[:,'cell_line'].value_counts()
min_group_size = min(group_counts)

keep_genes = n_passing_counts[n_passing_counts >= min_group_size].index.tolist()
print(f"Kept {len(keep_genes)} genes out of {counts.shape[1]}")

#Select just the passing genes
filt_counts = counts[keep_genes]

print(filt_counts.shape)

#Repeat the plot above using filtered data

#transpose the data and plot
temp =np.transpose(filt_counts+eps)

matplotlib.rcParams['figure.figsize'] = [12, 10]
temp.plot.hist(subplots=True, bins=logbins, layout=(4,3))
plt.xscale("log")
plt.show()





Above we see that we have removed the vast majority of low-expression genes for each sample, but there are some low-expression genes for each sample left in: so we haven't removed everything that is low in one or more conditions

So did we filter enough?  
Did we filter *too* much?

Two questions that can help frame our investigation:

1) what properties of gene expression data can we exploit to evaluate our filtering quantitatively?

2) what biology **can't** we study if we remove all the genes that were only detected in *some* samples?

We can begin to address the first question by considering a fundamental feature of RNAseq data, which is the relationship between expression levels and variance. (See slides for more details).

First we will log transform the data (since counts span several orders of magnitude, this will make it easier to visualize)

In [ ]:
#first calculate the mean expression for each gene (remember we did add a small count to zeroes)
mean = np.mean(np.log2(counts+eps),axis = 0)
mean


#next calculate the variability within each gene
sqrt_std = np.sqrt(np.log2(counts+eps).std())
sqrt_std

In [ ]:
#now plot the mean-variance as a scatterplot
plt.scatter(mean, sqrt_std, alpha=0.4,s = 1)
plt.title("Mean-Variance Plot (all genes)")
plt.xlabel("Mean Log2(counts)")
plt.ylabel("sqrt(Std.Dev.)")
plt.show()


A GREAT question here is 'what am i looking at?'

This is where we take advantage of the fact that taking the log of large values will compress the standard deviation.  We can see that at our highest count levels (on the right) the variance is the lowest.

If we follow the dots from right back to the left, we can see the points climbing.  This is another feature of taking the log of *low* counts: the variance is much larger when counts are low. (see slides for more details)

And if we go all the way to the left, the plot has a 'hook'.  

What does this mean?  

Remember that each gene in our unfiltered data set is represented in this plot - approximately 35,000.  If a gene *on average* has a low mean ***AND*** low variance, then it is a) likely undetected in most or all samples and b) provides very little meaningful data for downstream analysis.  After all, our aim is to identify genes whose expression differs by condition.  So if the gene is barely detected AND this is true across virtually all samples, then we can likely remove it without losing biologically useful information.   

And there is another factor at play here.

When we calculate DE genes, we'll be adjusting our p values for reasons you heard about in class recently.  If we were to run 35000 tests, our adjusted p values would be much higher than if we only perform 15000 tests.  This isn't p hacking - its simply making sure we only test genes with a potential to reveal differences between conditions.  

Now, what does this plot look like for our filtered genes?

In [ ]:
mean = np.mean(np.log2(filt_counts+eps),axis = 0)


sqrt_std = np.sqrt(np.log2(filt_counts+eps).std())

plt.scatter(mean, sqrt_std, alpha=0.4,s = 1)
plt.title("Mean-Variance Plot (filtered genes)")
plt.xlabel("Mean Log2(counts)")
plt.ylabel("sqrt(Std.Dev.)")
plt.show()


Interesting.  We've removed the huge spike of genes that were basically undetected, but we've clearly preserved some genes whose expression levels on average are very low.  

This will help ensure that we keep genes that are low-expression on average, but potentially more highly expressed in one group of interest, are included

# Normalizing RNAseq count data



## Sequencing depth and log transformation

A goal of normalization is to transform your RNAseq count data to make it look 'normal' i.e. the expression level of each gene roughly follows a normal distribution, with values symmetrically distributed around an average value, so that standard statistical approaches can be applied. So far, we have been using log-transformed data and adding a small count, for visualization purposes, but we can be more rigorous.

We also need to account for sequencing depth: i.e. the total number of reads quantified per-sample. A common approach is to use logCPM, or log2-transformed counts per-million total counts. This is a quick and dirty approach to making RNAseq data look 'normal'

In [ ]:
# Calculate total sample read depth
total_reads = counts.sum(axis = 1)

# plot read depth (with a trimmed y axis to emphasize the differences)
#All samples have between 30-40 million reads total
total_reads.plot(kind = "bar",color = "red")
plt.ylim(30e6,40e6)

total_reads_per_million=total_reads.divide(1e6)
#Now, convert to counts-per-million, log2 transform, and remove low-expression low-variance genes
log2_cpm = np.log2(counts.add(1).divide(total_reads_per_million, axis=0))[keep_genes]

log2_cpm.loc[:,keep_genes].head()

temp =np.transpose(log2_cpm)
temp.plot(kind='density')


In [ ]:
#Most highly expressed genes per sample
log2_cpm.reset_index().melt(id_vars='Sample Title', value_vars=keep_genes).groupby("Sample Title").apply(lambda x: x.nlargest(3, 'value'))


Highly expressed genes include:

 * **MT-CO1, MT-CO2**
      * Mitochondrial DNA-encoded subunits of cytochrome c oxidase (complex IV), essential for ATP production and often mutated in cancer.
 * **GAPDH, ENO1**
      * GAPDH (Glyceraldehyde-3-phosphate dehydrogenase) and ENO1 (Enolase 1) are key enzymes in glycolysis, often upregulated in cancer cells
 * **EEF1A1, EEF2**
      * EEF1A1 (Eukaryotic translation elongation factor 1 alpha 1) and EEF2 (Eukaryotic elongation factor 2) are part of protein translation machinery
 * **SERPINE2, FN1**
    * FN1 (Fibronectin 1) and SERPINE2 are involved in cell adhesion, migration, and tissue remodeling
 * **PMEL, TYRP1**
   * PMEL (Premelanosome Protein) and TYRP1 (Tyrosinase-Related Protein 1) are critical for melanosome biogenesis and pigment production
 * **VIM, ACTB**
   * VIM (Vimentin) and ACTB (Beta-actin) are key components of the cytoskeleton
  
A combination of basic metabolism, translation, cytoskeletal and melanocyte specific genes is consistent with what we might expect to be highly expressed in this melanoma cell line dataset.


## Comprehensive normalization with DESeq2

Comparing differences in gene expression between groups of samples uses standard statistical tools, such as linear models. However, using RNAseq counts directly is not appropriate, as we have to account for several factors that are indepedent of the real biological differences we want to measure. These include:


*   Differences in sequencing depth (i.e. the total number of reads obtained per sample)
*   Differences in RNA composition. RNAseq quantifies a fixed number of reads. A small number of very highly expressed reads in certain samples can skew expression of all other genes
* As we have already seen, the variance of a gene can depend on the mean expression of the gene. Standard linear models assume that variance is independent of the mean - normalization attempts to correct this relationship



Many of the differential expression packages (e.g. edgeR, limma-voom, Deseq2, etc) use different normalization strategies, however the basic principles are very similar. As a simplification, normalization applies some sort of 'scaling factor' correction to each sample and gene that attempts to bring sequencing data into line with necessary statistical assumptions. In our next lesson, we will be using the DESeq2 package for our differential expression calculations. In the standard Deseq2 pipeline, the normalization occurs behind the scenes.

Below is a short comparative example for how this looks.




In [ ]:

# Metadata data frame - the first few columns of the original counts csv file
meta = CountsData.iloc[:,0:5]

#Counts
raw_counts = CountsData[keep_genes]

#make sure the number of sample rows match between counts and metadata
meta.shape
raw_counts.shape

#Start by initializing the data, using the DeseqDataSet() function. The function requires 3 inputs:
# the CountsData data frame (containing the gene expression data in counts format),
# the meta data frame (containing the metadata associated with each sample),
# and the name of the column in meta that contains the group information for which comparison you want to make. To compare normal vs. metastatic, use "Stage"
dds = DeseqDataSet(counts = raw_counts, metadata = meta, design_factors="Stage")

#Run DESeq2 normalization
dds.fit_size_factors()
dds.fit_genewise_dispersions()

dds


#temp =np.transpose(log2_cpm)
#temp.plot(kind='density')


In [ ]:
#Calculate the DESeq2 normalized equivalent of the log2CPM data
dds.vst()


In [ ]:
#Let's plot the densities of the DESeq2 normalized data vs log2CPM
dds
dds_vst = pd.DataFrame(dds.layers['vst_counts'])
dds_vst.index=meta.index

np.transpose(dds_vst).plot(kind="density", title="DESeq2 vst")

np.transpose(log2_cpm).plot(kind='density', title="log2CPM")
